# Meta-Signal Q4 Gauntlet â€” Unsloth Fine-Tune

Fine-tunes **Llama-3.1-8B-Instruct** on expert demonstrations from the Q4 Gauntlet environment using [Unsloth](https://github.com/unslothai/unsloth) for 2Ã— faster training.

**Runtime**: Nvidia A10G Small (24 GB VRAM) â€” estimated **~12 min** for 10k records, 3 epochs.  
**Dataset**: `Anvit25/meta-signal-expert-demos` on HF Hub (10,250 Alpaca-format records).  
**Output**: LoRA adapter pushed to `Anvit25/meta-signal-q4-agent`.

## Steps
1. Install dependencies
2. Login + load dataset from HF Hub
3. Load Llama-3.1-8B with 4-bit QLoRA via Unsloth
4. Fine-tune with SFTTrainer
5. Inference test
6. Save adapter + push to HF Hub

In [ ]:
# â”€â”€ 1. Install â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
!pip install --quiet unsloth
!pip install --quiet trl datasets transformers accelerate bitsandbytes peft huggingface_hub
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
# â”€â”€ 2. Login + load dataset from HF Hub â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from huggingface_hub import login, hf_hub_download
import json, os

# Paste your HF write token (needs read+write scope)
# Get one at: https://huggingface.co/settings/tokens
HF_TOKEN = ""   # <-- paste your token here, or set HF_TOKEN env var
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
login(token=HF_TOKEN)

# Download dataset from HF Hub
DATA_REPO = "Anvit25/meta-signal-expert-demos"
DATA_PATH = hf_hub_download(
    repo_id   = DATA_REPO,
    filename  = "expert_demos.jsonl",
    repo_type = "dataset",
)
print(f"Dataset downloaded to: {DATA_PATH}")

records = [json.loads(l) for l in open(DATA_PATH, encoding="utf-8")]
print(f"Loaded {len(records):,} records")
phases = {}
for r in records:
    p = r['metadata']['phase']
    phases[p] = phases.get(p, 0) + 1
print("Phase distribution:", phases)

In [ ]:
# â”€â”€ Config (tuned for A10G Small 24 GB) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
MODEL_NAME   = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"  # pre-quantised
OUTPUT_DIR   = "meta-signal-lora"
HF_REPO      = "Anvit25/meta-signal-q4-agent"
MAX_SEQ_LEN  = 2048
LORA_RANK    = 16
LORA_ALPHA   = 32
BATCH_SIZE   = 8    # A10G can handle 8 vs T4's 4
GRAD_ACCUM   = 2    # effective batch = 16
EPOCHS       = 3
LR           = 2e-4
WARMUP_RATIO = 0.05

In [ ]:
# â”€â”€ 3. Load model with Unsloth â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_RANK,
    lora_alpha                 = LORA_ALPHA,
    target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                   "gate_proj", "up_proj", "down_proj"],
    lora_dropout               = 0,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 42,
)
print(model.print_trainable_parameters())

In [ ]:
# â”€â”€ Prepare dataset â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from datasets import Dataset

ALPACA_PROMPT = """Below is an instruction that describes a task, paired with an input \
that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

EOS_TOKEN = tokenizer.eos_token

def format_record(rec):
    text = ALPACA_PROMPT.format(
        instruction = rec["instruction"],
        input       = rec["input"],
        output      = rec["output"],
    ) + EOS_TOKEN
    return {"text": text}

dataset = Dataset.from_list([format_record(r) for r in records])
print(dataset)
print("\nSample (first 500 chars):\n", dataset[0]["text"][:500])

In [ ]:
# â”€â”€ 4. Fine-tune â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LEN,
    dataset_num_proc   = 4,
    packing            = True,   # pack short sequences for efficiency on A10G
    args = TrainingArguments(
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_ratio                = WARMUP_RATIO,
        num_train_epochs            = EPOCHS,
        learning_rate               = LR,
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),  # A10G supports bf16
        logging_steps               = 5,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "cosine",
        seed                        = 42,
        output_dir                  = OUTPUT_DIR,
        report_to                   = "none",
    ),
)

trainer_stats = trainer.train()
print(f"Training complete. Loss: {trainer_stats.training_loss:.4f}")

In [ ]:
# â”€â”€ 5. Quick inference test â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
FastLanguageModel.for_inference(model)

test_obs = """\
step=25 day=25 phase=Signal_Loss
budget_remaining=$8500.00 epsilon=16.500
privacy_regime=high_noise learning_status=Optimized
market_trend=Rising regulatory_violation=False
campaigns: camp_feed: spend=$0.0, noisy_conv=0.00, est_roas=0.000, ctr=0.0800, ci=[0.00,0.00] | \
camp_reels: spend=$0.0, noisy_conv=0.00, est_roas=0.000, ctr=0.0700, ci=[0.00,0.00] | \
camp_stories: spend=$0.0, noisy_conv=0.00, est_roas=0.000, ctr=0.0600, ci=[0.00,0.00]"""

prompt = ALPACA_PROMPT.format(
    instruction = (
        "You are an advertising budget optimisation agent in Phase 2 (days 21-50). "
        "iOS ATT has caused 3x noise. Use use_capi=true sparingly. "
        "Hold Phase 1 allocation steady. Output a JSON action."
    ),
    input  = test_obs,
    output = "",
)

inputs  = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.1, do_sample=True)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)[len(prompt):]
print("Model output:")
print(response)

In [ ]:
# â”€â”€ 6. Save adapter + push to HF Hub â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to ./{OUTPUT_DIR}")

model.push_to_hub(HF_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
print(f"Pushed to https://huggingface.co/{HF_REPO}")
print()
print("Training summary:")
print(f"  Loss:    {trainer_stats.training_loss:.4f}")
print(f"  Runtime: {trainer_stats.metrics.get('train_runtime', 0)/60:.1f} min")
print(f"  Steps:   {trainer_stats.global_step}")

## Next Steps

1. **Evaluate** the fine-tuned model against the live environment:
   ```bash
   python inference.py --task 7 --model Anvit25/meta-signal-q4-agent
   ```
2. **Compare** score against the ExpertBot baseline (~0.66 on Task 7).
3. **Iterate** â€” generate 500 episodes and re-train for higher quality.
4. **RLHF loop** (optional): use the grader score as reward signal for PPO.